# 08 - Pydantic Deep Dive & API-Adjacent Python Concepts

Concise, interview/revision focused. Continues from `07-api-fundamentals.ipynb`. Setup below installs everything needed.

In [1]:
import importlib.util, subprocess, sys
for pkg in ["fastapi", "uvicorn", "pydantic", "pydantic_settings", "jwt", "sqlalchemy", "httpx2"]:
    mod = "pyjwt" if pkg == "jwt" else pkg
    if importlib.util.find_spec(pkg) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", mod], check=True)
import fastapi, pydantic
print("fastapi", fastapi.__version__, "| pydantic", pydantic.__version__)

fastapi 0.141.1 | pydantic 2.13.4


# Part 1 - Pydantic Deep Dive

**Interview Q: why Pydantic over manual validation?** Declarative (type hints ARE the schema), auto docs, nested/reusable models, fast (Rust core in v2), integrates natively with FastAPI's 422 error handling.

### Field constraints, nested models, Optional, aliases, defaults

In [2]:
from pydantic import BaseModel, Field, field_validator
from typing import Optional

class Address(BaseModel):
    city: str
    zip_code: str = Field(alias="zipCode")           # camelCase JSON -> snake_case Python

class User(BaseModel):
    name: str
    age: int = Field(gt=0, le=120)                    # numeric bounds
    email: Optional[str] = None                        # optional, defaults to None
    role: str = "user"                                  # default value
    address: Address                                    # nested model
    model_config = {"populate_by_name": True}

u = User(name="Ada", age=30, address={"city": "London", "zipCode": "E1"})
print(u.model_dump())
print(u.model_dump_json())

{'name': 'Ada', 'age': 30, 'email': None, 'role': 'user', 'address': {'city': 'London', 'zip_code': 'E1'}}
{"name":"Ada","age":30,"email":null,"role":"user","address":{"city":"London","zip_code":"E1"}}


### Custom validators -- logic that Field() alone cannot express

In [3]:
class SignupRequest(BaseModel):
    username: str
    password: str

    @field_validator("username")
    @classmethod
    def username_alphanumeric(cls, v):
        if not v.isalnum():
            raise ValueError("username must be alphanumeric")
        return v

try:
    SignupRequest(username="bad name!", password="x")
except Exception as e:
    print("rejected:", str(e).splitlines()[0])

print(SignupRequest(username="ada123", password="x"))

rejected: 1 validation error for SignupRequest
username='ada123' password='x'


### pydantic-settings: typed config from environment variables

The standard way to load config (DB URLs, API keys, feature flags) with validation, instead of raw `os.environ.get(...)` calls scattered everywhere.

In [4]:
import os
from pydantic_settings import BaseSettings

os.environ["APP_DEBUG"] = "true"   # normally set outside Python, e.g. in the shell or a .env file

class Settings(BaseSettings):
    app_name: str = "ml-service"
    debug: bool = False
    max_batch_size: int = 32

settings = Settings()   # DEBUG picked up from env automatically (case-insensitive by default)
print(settings.model_dump())

{'app_name': 'ml-service', 'debug': False, 'max_batch_size': 32}


# Part 2 - Auth: API Keys and JWT

In [5]:
from fastapi import FastAPI, Depends, Header, HTTPException
import jwt, datetime

app = FastAPI()
SECRET = "demo-secret"

def require_api_key(x_api_key: str = Header(...)):
    if x_api_key != "secret-key-123":
        raise HTTPException(401, "invalid API key")
    return x_api_key

@app.get("/secure", dependencies=[Depends(require_api_key)])
def secure():
    return {"ok": True}

def require_jwt(authorization: str = Header(...)):
    token = authorization.removeprefix("Bearer ").strip()
    try:
        return jwt.decode(token, SECRET, algorithms=["HS256"])
    except jwt.ExpiredSignatureError:
        raise HTTPException(401, "token expired")
    except jwt.InvalidTokenError:
        raise HTTPException(401, "invalid token")

@app.get("/me")
def me(payload: dict = Depends(require_jwt)):
    return {"user": payload["sub"]}

token = jwt.encode({"sub": "u1", "exp": datetime.datetime.now(datetime.timezone.utc) + datetime.timedelta(minutes=5)}, SECRET, algorithm="HS256")
print("token parts:", token.count(".") + 1)

token parts: 3


# Part 3 - Logging & Middleware

`print` cannot be filtered by severity, redirected to a log aggregator, or turned off in production without editing code. `logging` can. Middleware runs around every request/response -- must be registered before the app handles its first request, which is why this comes before any testing below.

In [6]:
import logging

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(name)s: %(message)s")
logger = logging.getLogger("ml_service")

@app.middleware("http")
async def log_requests(request, call_next):
    logger.info(f"{request.method} {request.url.path}")
    response = await call_next(request)
    logger.info(f"-> {response.status_code}")
    return response

logger.info("app configured")

2026-07-31 11:49:38,746 INFO ml_service: app configured


# Part 4 - CORS

Browsers block cross-origin JS requests unless the server explicitly allows them -- required the moment a frontend on one domain calls an API on another. CORS is enforced by the BROWSER, not the server: a non-browser client, such as `requests` or a server-to-server call, ignores it entirely -- it is not a security boundary by itself. CORS is just a pre-built middleware, same rule as above: register it before the first request.

In [7]:
from fastapi.middleware.cors import CORSMiddleware

app.add_middleware(
    CORSMiddleware,
    allow_origins=["https://myfrontend.com"],
    allow_methods=["GET", "POST"],
    allow_headers=["*"],
)
print("middleware registered:", [m.cls.__name__ for m in app.user_middleware])

middleware registered: ['CORSMiddleware', 'BaseHTTPMiddleware']


# Part 5 - Testing FastAPI Apps: pytest + TestClient

TestClient calls the ASGI app directly (no real socket/port needed) -- the standard way to unit test an API. In a real project this code lives in `test_*.py` files run via `pytest`; here it runs inline for demonstration. This is also the app's first request, so it must come after all middleware/route registration above.

In [8]:
from fastapi.testclient import TestClient

client = TestClient(app)

def test_secure_requires_key():
    assert client.get("/secure").status_code == 422        # missing required header -> FastAPI validation, not our 401
    assert client.get("/secure", headers={"x-api-key": "wrong"}).status_code == 401   # present but wrong -> our check
    assert client.get("/secure", headers={"x-api-key": "secret-key-123"}).status_code == 200

def test_me_with_valid_token():
    r = client.get("/me", headers={"Authorization": f"Bearer {token}"})
    assert r.status_code == 200
    assert r.json()["user"] == "u1"

test_secure_requires_key()
test_me_with_valid_token()
print("tests passed")   # pytest would discover/run these automatically via `pytest test_app.py`

2026-07-31 11:49:38,935 INFO ml_service: GET /secure


2026-07-31 11:49:38,941 INFO ml_service: -> 422


2026-07-31 11:49:38,944 INFO httpx2: HTTP Request: GET http://testserver/secure "HTTP/1.1 422 Unprocessable Entity"


2026-07-31 11:49:38,948 INFO ml_service: GET /secure


2026-07-31 11:49:38,955 INFO ml_service: -> 401


...

2026-07-31 11:49:38,969 INFO httpx2: HTTP Request: GET http://testserver/secure "HTTP/1.1 200 OK"


2026-07-31 11:49:38,972 INFO ml_service: GET /me


2026-07-31 11:49:38,979 INFO ml_service: -> 200


2026-07-31 11:49:38,982 INFO httpx2: HTTP Request: GET http://testserver/me "HTTP/1.1 200 OK"


tests passed


# Part 6 - Background Tasks & WebSockets

`BackgroundTasks` run AFTER the response is sent -- good for logging/notifications that should not add to response latency. WebSockets keep a persistent bidirectional connection -- used for streaming tokens from an LLM, live progress updates, etc., where request/response HTTP does not fit.

**Interview Q: BackgroundTasks vs Celery?** BackgroundTasks runs in-process and dies if the server restarts -- fine for fire-and-forget logging, wrong for anything that must survive a crash or needs retries/scheduling (use Celery/RQ/a real queue for that).

In [9]:
from fastapi import BackgroundTasks, WebSocket

def send_notification(msg: str):
    print("background task ran:", msg)   # e.g. would email/log/write to a queue

@app.post("/notify")
def notify(background_tasks: BackgroundTasks):
    background_tasks.add_task(send_notification, "job finished")
    return {"status": "accepted"}   # client gets this immediately, task runs after

@app.websocket("/ws")
async def websocket_endpoint(ws: WebSocket):
    await ws.accept()
    for i in range(3):
        await ws.send_json({"progress": i})
    await ws.close()

with TestClient(app) as c:
    print(c.post("/notify").json())
    with c.websocket_connect("/ws") as ws:
        for _ in range(3):
            print(ws.receive_json())

2026-07-31 11:49:39,002 INFO ml_service: POST /notify


2026-07-31 11:49:39,009 INFO ml_service: -> 200


2026-07-31 11:49:39,012 INFO httpx2: HTTP Request: POST http://testserver/notify "HTTP/1.1 200 OK"


background task ran: job finished
{'status': 'accepted'}
{'progress': 0}
{'progress': 1}
{'progress': 2}


# Part 7 - Database Basics: SQLAlchemy + FastAPI

The standard pattern: engine (connection), Session per request via `Depends`, ORM model, CRUD. SQLite in-memory here so it runs with zero setup; swapping to Postgres is just a different connection URL.

**Interview Q: why a DB session per request, not one global session?** Avoids sharing one connection/transaction across concurrent requests. `Depends(get_db)` + `yield` guarantees `close()` runs even on error -- the same contextmanager guarantee from earlier notebooks.

In [10]:
from sqlalchemy import create_engine, Column, Integer, String
from sqlalchemy.orm import declarative_base, sessionmaker
from sqlalchemy.pool import StaticPool

# StaticPool pins every connection to the SAME in-memory DB -- without it, each new
# connection (e.g. each request's Session) would see a separate, empty :memory: database.
engine = create_engine("sqlite:///:memory:", connect_args={"check_same_thread": False}, poolclass=StaticPool)
SessionLocal = sessionmaker(bind=engine)
Base = declarative_base()

class Prediction(Base):
    __tablename__ = "predictions"
    id = Column(Integer, primary_key=True)
    input_summary = Column(String)
    result = Column(String)

Base.metadata.create_all(engine)

def get_db():
    db = SessionLocal()
    try:
        yield db
    finally:
        db.close()

@app.post("/log-prediction")
def log_prediction(input_summary: str, result: str, db=Depends(get_db)):
    row = Prediction(input_summary=input_summary, result=result)
    db.add(row)
    db.commit()
    db.refresh(row)
    return {"id": row.id}

with TestClient(app) as c:
    r = c.post("/log-prediction", params={"input_summary": "x=[1,2]", "result": "0.83"})
    print(r.json())

db = SessionLocal()
print([(p.id, p.input_summary, p.result) for p in db.query(Prediction).all()])
db.close()

2026-07-31 11:49:40,191 INFO ml_service: POST /log-prediction


2026-07-31 11:49:40,218 INFO ml_service: -> 200


2026-07-31 11:49:40,221 INFO httpx2: HTTP Request: POST http://testserver/log-prediction?input_summary=x%3D%5B1%2C2%5D&result=0.83 "HTTP/1.1 200 OK"


{'id': 1}
[(1, 'x=[1,2]', '0.83')]


## Interview rapid-fire

- Pydantic v1 vs v2: v2 has a Rust core (5-50x faster), `model_dump()`/`model_dump_json()` replace `.dict()`/`.json()`, `field_validator` replaces `validator`.
- Middleware and routes differ in one important way: routes can be added to a FastAPI app even after it has started serving; middleware cannot -- it must be registered before the first request.
- `Header(...)` with no default makes a header required; a missing one is rejected by FastAPI's automatic validation (422) before your function body, and your own auth logic, ever runs.

## Practice

1. Add `@field_validator` to `Settings.max_batch_size` rejecting values over 256.
2. Add a `GET /predictions` endpoint returning all logged `Prediction` rows.
3. Add a background task to `log_prediction` that "notifies" after logging, without slowing the response.